# ticker_attention_diagnostics

Diagnostics for the thesis question: do some tickers have too little Google, Reddit, or GDELT attention for alternative-data features to be useful?

The notebook first measures raw attention coverage by ticker, then compares compact logistic-regression models against a price+volume baseline at ticker level. It reports both pooled-model per-ticker lifts and ticker-specific model lifts.


In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)


In [2]:
from __future__ import annotations

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG = {
    "dataset_path": PROJECT_ROOT / "data" / "datasets" / "stock_panel_nine_tickers_session_aligned_full_history_adjusted_google_score_raw.csv",
    "excluded_tickers": ["NFLX"],
    "neutral_band": 0.005,
    "test_size": 0.25,
    "validation_fraction_within_pretest": 0.25,
    "min_validation_dates": 20,
    "gap_days": 1,
    "random_state": 42,
    "threshold_min_quantile": 0.05,
    "threshold_max_quantile": 0.95,
    "threshold_grid_size": 101,
    "min_ticker_train_rows": 300,
    "min_ticker_validation_rows": 80,
    "min_ticker_test_rows": 80,
}

LOGREG_PARAM_GRID = [
    {
        "param_set": "logreg_l2_C0p1_balanced",
        "penalty": "l2",
        "C": 0.1,
        "solver": "lbfgs",
        "class_weight": "balanced",
        "max_iter": 3000,
    },
    {
        "param_set": "logreg_l2_C1_balanced",
        "penalty": "l2",
        "C": 1.0,
        "solver": "lbfgs",
        "class_weight": "balanced",
        "max_iter": 3000,
    },
    {
        "param_set": "logreg_l1_C0p1_balanced",
        "penalty": "l1",
        "C": 0.1,
        "solver": "liblinear",
        "class_weight": "balanced",
        "max_iter": 3000,
    },
    {
        "param_set": "logreg_l2_C0p1_unweighted",
        "penalty": "l2",
        "C": 0.1,
        "solver": "lbfgs",
        "class_weight": None,
        "max_iter": 3000,
    },
]

PRICE_FEATURES = [
    "return_1d",
    "return_5d",
    "return_20d",
    "rolling_volatility_20d",
]

BASELINE_FEATURE_SET = "price + volume"
FEATURE_SETS = {
    "price only": PRICE_FEATURES,
    "price + volume": PRICE_FEATURES + ["volume_zscore_20d"],
    "price + volume + GDELT full": PRICE_FEATURES
    + ["volume_zscore_20d", "gdelt_sentiment_zscore_10d", "gdelt_article_count_zscore_6d"],
    "price + volume + Reddit full": PRICE_FEATURES
    + ["volume_zscore_20d", "reddit_sentiment_zscore_6d", "reddit_comment_count_zscore_6d"],
    "price + volume + Google attention": PRICE_FEATURES + ["volume_zscore_20d", "google_trends_zscore_10d"],
    "price + volume + GDELT attention": PRICE_FEATURES + ["volume_zscore_20d", "gdelt_article_count_zscore_6d"],
    "price + volume + Reddit attention": PRICE_FEATURES + ["volume_zscore_20d", "reddit_comment_count_zscore_6d"],
    "price + volume + all attention": PRICE_FEATURES
    + ["volume_zscore_20d", "gdelt_article_count_zscore_6d", "reddit_comment_count_zscore_6d", "google_trends_zscore_10d"],
    "price + volume + alt sentiment compact": PRICE_FEATURES
    + ["volume_zscore_20d", "gdelt_sentiment_zscore_10d", "reddit_sentiment_zscore_6d", "google_trends_zscore_10d"],
}

FEATURE_FAMILIES = {
    "price only": "price only",
    "price + volume": "price + volume",
    "price + volume + GDELT full": "price + volume + GDELT",
    "price + volume + Reddit full": "price + volume + Reddit",
    "price + volume + Google attention": "price + volume + Google attention",
    "price + volume + GDELT attention": "price + volume + GDELT attention",
    "price + volume + Reddit attention": "price + volume + Reddit attention",
    "price + volume + all attention": "price + volume + all attention",
    "price + volume + alt sentiment compact": "price + volume + compact alternative sentiment",
}

FEATURE_SETS_TO_TEST = list(FEATURE_SETS)
ATTENTION_FEATURE_SETS = [
    feature_set for feature_set in FEATURE_SETS_TO_TEST if "attention" in FEATURE_FAMILIES[feature_set]
]

pd.DataFrame(LOGREG_PARAM_GRID)


,param_set,penalty,C,solver,class_weight,max_iter
0,logreg_l2_C0p1_balanced,l2,0.1,lbfgs,balanced,3000
1,logreg_l2_C1_balanced,l2,1.0,lbfgs,balanced,3000
2,logreg_l1_C0p1_balanced,l1,0.1,liblinear,balanced,3000
3,logreg_l2_C0p1_unweighted,l2,0.1,lbfgs,NaN,3000


In [3]:
from __future__ import annotations

def transformed_source_series(
    frame: pd.DataFrame,
    source_col: str,
    value_transform: str | None = None,
) -> pd.Series:
    values = frame[source_col].astype(float)
    if value_transform is None:
        return values
    if value_transform == "log1p":
        return np.log1p(values.clip(lower=0.0))
    raise ValueError(f"Unsupported value_transform: {value_transform}")


def add_trailing_zscore(
    frame: pd.DataFrame,
    source_col: str,
    output_col: str,
    window: int,
    *,
    value_transform: str | None = None,
    clip_value: float | None = None,
) -> None:
    values = transformed_source_series(frame, source_col, value_transform)
    grouped = values.groupby(frame["ticker"])
    rolling_mean = grouped.transform(lambda s: s.shift(1).rolling(window).mean())
    rolling_std = grouped.transform(lambda s: s.shift(1).rolling(window).std())
    zscore = (values - rolling_mean) / rolling_std.replace(0.0, np.nan)
    if clip_value is not None:
        zscore = zscore.clip(lower=-clip_value, upper=clip_value)
    frame[output_col] = zscore


def build_feature_frame(raw_df: pd.DataFrame, neutral_band: float) -> pd.DataFrame:
    frame = raw_df.copy().sort_values(["ticker", "date"]).reset_index(drop=True)
    frame["reddit_sentiment_missing"] = frame["comm_reddit_vader_mean"].isna().astype(float)
    frame["gdelt_sentiment_missing"] = frame["gdelt_sentiment_score"].isna().astype(float)
    frame["comm_reddit_posts"] = frame["comm_reddit_posts"].fillna(0.0)
    frame["gdelt_articles"] = frame["gdelt_articles"].fillna(0.0)

    price_group = frame.groupby("ticker")["stock_price"]
    frame["return_1d"] = price_group.pct_change(1)
    frame["return_5d"] = price_group.pct_change(5)
    frame["return_20d"] = price_group.pct_change(20)
    frame["rolling_volatility_20d"] = (
        frame.groupby("ticker")["return_1d"].transform(lambda s: s.shift(1).rolling(20).std())
    )

    add_trailing_zscore(frame, "stock_volume", "volume_zscore_20d", 20)
    add_trailing_zscore(frame, "gdelt_sentiment_score", "gdelt_sentiment_zscore_10d", 10)
    add_trailing_zscore(frame, "gdelt_articles", "gdelt_article_count_zscore_6d", 6)
    add_trailing_zscore(frame, "comm_reddit_vader_mean", "reddit_sentiment_zscore_6d", 6)
    add_trailing_zscore(frame, "comm_reddit_posts", "reddit_comment_count_zscore_6d", 6)
    add_trailing_zscore(frame, "google_trends_score", "google_trends_zscore_10d", 10)

    frame["future_return_1d"] = price_group.shift(-1) / frame["stock_price"] - 1.0
    frame["target"] = np.select(
        [
            frame["future_return_1d"] < -neutral_band,
            frame["future_return_1d"] > neutral_band,
        ],
        [0, 1],
        default=np.nan,
    )
    frame["target_available"] = frame["future_return_1d"].notna()
    frame["is_neutral"] = frame["target_available"] & frame["future_return_1d"].abs().le(neutral_band)
    return frame


def make_split_dates(
    frame: pd.DataFrame,
    test_size: float,
    validation_fraction_within_pretest: float,
    min_validation_dates: int,
    gap_days: int,
) -> tuple[list[pd.Timestamp], list[pd.Timestamp], list[pd.Timestamp]]:
    unique_dates = sorted(frame["date"].drop_duplicates())
    test_start_idx = int(np.floor(len(unique_dates) * (1.0 - test_size)))
    test_start_idx = min(max(test_start_idx, 2), len(unique_dates) - 1)
    pretest_end_idx = max(test_start_idx - gap_days, 1)
    pretest_dates = unique_dates[:pretest_end_idx]

    validation_size = int(np.floor(len(pretest_dates) * validation_fraction_within_pretest))
    validation_size = max(min_validation_dates, validation_size)
    validation_size = min(max(validation_size, 1), len(pretest_dates) - 1)

    validation_start_idx = len(pretest_dates) - validation_size
    train_end_idx = max(validation_start_idx - gap_days, 1)

    train_dates = unique_dates[:train_end_idx]
    validation_dates = pretest_dates[validation_start_idx:]
    test_dates = unique_dates[test_start_idx:]
    return train_dates, validation_dates, test_dates


def subset_by_dates(frame: pd.DataFrame, dates: list[pd.Timestamp]) -> pd.DataFrame:
    return frame[frame["date"].isin(dates)].copy()


def safe_auc(y_true: pd.Series, scores: np.ndarray) -> float:
    if pd.Series(y_true).nunique() < 2:
        return np.nan
    return float(roc_auc_score(y_true, scores))


def safe_balanced_accuracy(y_true: pd.Series, preds: np.ndarray) -> float:
    if pd.Series(y_true).nunique() < 2:
        return np.nan
    return float(balanced_accuracy_score(y_true, preds))


In [4]:
from __future__ import annotations

raw_df = pd.read_csv(CONFIG["dataset_path"], parse_dates=["date"])
raw_df = raw_df[~raw_df["ticker"].isin(CONFIG["excluded_tickers"])].copy()
raw_df = raw_df.sort_values(["ticker", "date"]).reset_index(drop=True)

feature_df = build_feature_frame(raw_df, neutral_band=CONFIG["neutral_band"])
train_dates, validation_dates, test_dates = make_split_dates(
    feature_df,
    test_size=CONFIG["test_size"],
    validation_fraction_within_pretest=CONFIG["validation_fraction_within_pretest"],
    min_validation_dates=CONFIG["min_validation_dates"],
    gap_days=CONFIG["gap_days"],
)

split_date_map = {"train": train_dates, "validation": validation_dates, "test": test_dates}
feature_df["split"] = "gap"
for split_name, split_dates in split_date_map.items():
    feature_df.loc[feature_df["date"].isin(split_dates), "split"] = split_name

modeled_df = feature_df[feature_df["target"].isin([0.0, 1.0])].copy()
modeled_df["target"] = modeled_df["target"].astype(int)

train_df = subset_by_dates(modeled_df, train_dates)
validation_df = subset_by_dates(modeled_df, validation_dates)
test_df = subset_by_dates(modeled_df, test_dates)

split_summary_df = (
    modeled_df[modeled_df["split"].isin(["train", "validation", "test"])]
    .groupby("split")
    .agg(
        modeled_rows=("target", "size"),
        dates=("date", "nunique"),
        positive_rate=("target", "mean"),
        date_min=("date", "min"),
        date_max=("date", "max"),
    )
    .reindex(["train", "validation", "test"])
    .reset_index()
)

ticker_split_summary_df = (
    modeled_df[modeled_df["split"].isin(["train", "validation", "test"])]
    .groupby(["ticker", "split"])
    .agg(
        modeled_rows=("target", "size"),
        dates=("date", "nunique"),
        positive_rate=("target", "mean"),
    )
    .reset_index()
)

split_summary_df


,split,modeled_rows,dates,positive_rate,date_min,date_max
0,train,4456,704,0.511670,2021-01-04,2023-10-19
1,validation,1424,235,0.568118,2023-10-23,2024-09-27
2,test,1886,313,0.526511,2024-10-01,2025-12-30


In [5]:
from __future__ import annotations

ATTENTION_SIGNAL_COLUMNS = {
    "google": "google_trends_score",
    "reddit": "comm_reddit_posts",
    "gdelt": "gdelt_articles",
}


def attention_coverage_summary(frame: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    rows = []
    for group_key, group in frame.groupby(group_cols, dropna=False):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        row = dict(zip(group_cols, group_key))
        row["rows"] = len(group)
        for signal_name, column in ATTENTION_SIGNAL_COLUMNS.items():
            values = pd.to_numeric(group[column], errors="coerce")
            row[f"{signal_name}_available_rate"] = float(values.notna().mean())
            row[f"{signal_name}_nonzero_rate"] = float(values.fillna(0.0).gt(0.0).mean())
            row[f"{signal_name}_median"] = float(values.median())
            row[f"{signal_name}_mean"] = float(values.mean())
            row[f"{signal_name}_p90"] = float(values.quantile(0.90))
            row[f"{signal_name}_total"] = float(values.fillna(0.0).sum())
        rows.append(row)
    return pd.DataFrame(rows)


attention_coverage_by_ticker_df = attention_coverage_summary(feature_df, ["ticker"])

for signal_name in ATTENTION_SIGNAL_COLUMNS:
    attention_coverage_by_ticker_df[f"{signal_name}_rank"] = attention_coverage_by_ticker_df[
        f"{signal_name}_mean"
    ].rank(ascending=False, method="min")

attention_coverage_by_ticker_df["avg_attention_rank"] = attention_coverage_by_ticker_df[
    ["google_rank", "reddit_rank", "gdelt_rank"]
].mean(axis=1)
attention_coverage_by_ticker_df["low_overall_attention_flag"] = (
    attention_coverage_by_ticker_df["avg_attention_rank"]
    >= attention_coverage_by_ticker_df["avg_attention_rank"].quantile(0.75)
)
attention_coverage_by_ticker_df["weakest_attention_source"] = attention_coverage_by_ticker_df[
    ["google_rank", "reddit_rank", "gdelt_rank"]
].idxmax(axis=1).str.replace("_rank", "", regex=False)

total_columns = [f"{signal_name}_total" for signal_name in ATTENTION_SIGNAL_COLUMNS]
for total_column in total_columns:
    attention_coverage_by_ticker_df[total_column.replace("_total", "_share")] = (
        attention_coverage_by_ticker_df[total_column] / attention_coverage_by_ticker_df[total_column].sum()
    )

attention_coverage_by_ticker_split_df = attention_coverage_summary(
    feature_df[feature_df["split"].isin(["train", "validation", "test"])],
    ["ticker", "split"],
)

attention_coverage_display_columns = [
    "ticker",
    "rows",
    "google_mean",
    "reddit_mean",
    "gdelt_mean",
    "google_rank",
    "reddit_rank",
    "gdelt_rank",
    "avg_attention_rank",
    "weakest_attention_source",
    "low_overall_attention_flag",
]

attention_coverage_by_ticker_df[attention_coverage_display_columns].sort_values(
    ["low_overall_attention_flag", "avg_attention_rank"],
    ascending=[False, False],
).reset_index(drop=True)


,ticker,rows,google_mean,reddit_mean,gdelt_mean,google_rank,reddit_rank,gdelt_rank,avg_attention_rank,weakest_attention_source,low_overall_attention_flag
0,AMD,1255,21.512428,30.303586,0.020628,7.0,7.0,8.0,7.333333,gdelt,True
1,NVDA,1255,17.126982,58.322709,0.158757,8.0,4.0,5.0,5.666667,google,True
2,AMZN,1255,31.598097,54.796813,0.085730,4.0,5.0,6.0,5.000000,gdelt,False
3,META,1255,22.519854,24.498805,3.054745,6.0,8.0,1.0,5.000000,reddit,False
4,AAPL,1255,35.245331,81.182470,0.045539,2.0,2.0,7.0,3.666667,gdelt,False
5,MSFT,1255,43.078871,48.926693,0.567273,1.0,6.0,3.0,3.333333,reddit,False
6,TSLA,1255,26.664668,122.027888,0.554015,5.0,1.0,4.0,3.333333,google,False
7,GOOGL,1255,34.146333,70.502789,1.337674,3.0,3.0,2.0,2.666667,google,False


In [6]:
from __future__ import annotations

missing_feature_columns = sorted(
    {
        feature
        for feature_set in FEATURE_SETS_TO_TEST
        for feature in FEATURE_SETS[feature_set]
        if feature not in feature_df.columns
    }
)
if missing_feature_columns:
    raise KeyError(f"Missing feature columns: {missing_feature_columns}")

candidate_feature_sets_df = pd.DataFrame(
    [
        {
            "feature_set": feature_set,
            "feature_family": FEATURE_FAMILIES[feature_set],
            "n_features": len(features),
            "is_attention_set": feature_set in ATTENTION_FEATURE_SETS,
            "features": features,
        }
        for feature_set, features in FEATURE_SETS.items()
    ]
)

print(f"Tickers: {modeled_df['ticker'].nunique()}")
print(f"Feature sets to test: {len(FEATURE_SETS_TO_TEST)}")
print(f"Attention feature sets: {len(ATTENTION_FEATURE_SETS)}")
print(f"Logistic-regression parameter sets: {len(LOGREG_PARAM_GRID)}")
print(
    "Ticker-specific fits: "
    f"{modeled_df['ticker'].nunique() * len(FEATURE_SETS_TO_TEST) * len(LOGREG_PARAM_GRID)}"
)

candidate_feature_sets_df


Tickers: 8
Feature sets to test: 9
Attention feature sets: 4
Logistic-regression parameter sets: 4
Ticker-specific fits: 288


,feature_set,feature_family,n_features,is_attention_set,features
0,price only,price only,4,False,"[return_1d, return_5d, return_20d, rolling_vol..."
1,price + volume,price + volume,5,False,"[return_1d, return_5d, return_20d, rolling_vol..."
2,price + volume + GDELT full,price + volume + GDELT,7,False,"[return_1d, return_5d, return_20d, rolling_vol..."
3,price + volume + Reddit full,price + volume + Reddit,7,False,"[return_1d, return_5d, return_20d, rolling_vol..."
4,price + volume + Google attention,price + volume + Google attention,6,True,"[return_1d, return_5d, return_20d, rolling_vol..."
5,price + volume + GDELT attention,price + volume + GDELT attention,6,True,"[return_1d, return_5d, return_20d, rolling_vol..."
6,price + volume + Reddit attention,price + volume + Reddit attention,6,True,"[return_1d, return_5d, return_20d, rolling_vol..."
7,price + volume + all attention,price + volume + all attention,8,True,"[return_1d, return_5d, return_20d, rolling_vol..."
8,price + volume + alt sentiment compact,price + volume + compact alternative sentiment,8,False,"[return_1d, return_5d, return_20d, rolling_vol..."


In [7]:
from __future__ import annotations

LOGREG_PARAM_COLUMNS = ["penalty", "C", "solver", "class_weight", "max_iter"]


def build_logreg_pipeline_from_params(params: dict) -> Pipeline:
    model_params = dict(params)
    model_params.pop("param_set", None)
    model_params.setdefault("random_state", CONFIG["random_state"])
    return Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(**model_params)),
        ]
    )


def candidate_thresholds_from_scores(scores: np.ndarray) -> np.ndarray:
    finite_scores = np.asarray(scores, dtype=float)
    finite_scores = finite_scores[np.isfinite(finite_scores)]
    if len(finite_scores) == 0:
        return np.array([0.0])
    quantiles = np.linspace(
        CONFIG["threshold_min_quantile"],
        CONFIG["threshold_max_quantile"],
        CONFIG["threshold_grid_size"],
    )
    thresholds = np.quantile(finite_scores, quantiles)
    return np.unique(np.r_[thresholds, 0.0])


def best_threshold_for_balanced_accuracy(y_true: pd.Series, scores: np.ndarray) -> tuple[float, float]:
    rows = []
    for threshold in candidate_thresholds_from_scores(scores):
        preds = (scores >= threshold).astype(int)
        rows.append((float(threshold), safe_balanced_accuracy(y_true, preds)))
    return max(rows, key=lambda item: -np.inf if pd.isna(item[1]) else item[1])


def metrics_from_scores(y_true: pd.Series, scores: np.ndarray, threshold: float) -> dict:
    preds = (scores >= threshold).astype(int)
    return {
        "preds": preds,
        "predicted_positive_rate": float(np.mean(preds)),
        "accuracy": float(accuracy_score(y_true, preds)),
        "balanced_accuracy": safe_balanced_accuracy(y_true, preds),
        "auc": safe_auc(y_true, scores),
    }


def add_param_columns(row: dict, params: dict) -> None:
    for column in LOGREG_PARAM_COLUMNS:
        row[column] = params.get(column)


def evaluate_logreg_params(
    *,
    feature_set_name: str,
    params: dict,
    train_input_df: pd.DataFrame,
    eval_input_df: pd.DataFrame,
    split_name: str,
    scope: str,
    ticker: str | None = None,
    decision_threshold: float | None = None,
    tune_threshold: bool = False,
    return_predictions: bool = False,
) -> dict | tuple[dict, pd.DataFrame]:
    features = FEATURE_SETS[feature_set_name]
    pipeline = build_logreg_pipeline_from_params(params)
    pipeline.fit(train_input_df[features], train_input_df["target"])
    scores = pipeline.decision_function(eval_input_df[features])

    threshold_source = "fixed"
    if tune_threshold:
        decision_threshold, threshold_selection_balanced_accuracy = best_threshold_for_balanced_accuracy(
            eval_input_df["target"],
            scores,
        )
        threshold_source = "validation_tuned"
    else:
        threshold_selection_balanced_accuracy = np.nan
        if decision_threshold is None:
            decision_threshold = 0.0
            threshold_source = "default_zero"

    metric_result = metrics_from_scores(eval_input_df["target"], scores, float(decision_threshold))
    preds = metric_result.pop("preds")
    coefficients = pipeline.named_steps["model"].coef_.ravel()
    row = {
        "scope": scope,
        "ticker": ticker if ticker is not None else "ALL",
        "split": split_name,
        "feature_set": feature_set_name,
        "feature_family": FEATURE_FAMILIES[feature_set_name],
        "is_attention_set": feature_set_name in ATTENTION_FEATURE_SETS,
        "features": ", ".join(features),
        "param_set": params["param_set"],
        "n_features": len(features),
        "train_rows": len(train_input_df),
        "eval_rows": len(eval_input_df),
        "eval_positive_rate": float(eval_input_df["target"].mean()),
        "decision_threshold": float(decision_threshold),
        "threshold_source": threshold_source,
        "threshold_selection_balanced_accuracy": threshold_selection_balanced_accuracy,
        "n_iter": int(np.max(pipeline.named_steps["model"].n_iter_)),
        "n_nonzero_coefficients": int(np.count_nonzero(np.abs(coefficients) > 1e-8)),
        **metric_result,
    }
    add_param_columns(row, params)

    if not return_predictions:
        return row

    predictions_df = eval_input_df[["date", "ticker", "target"]].copy()
    predictions_df["scope"] = scope
    predictions_df["feature_set"] = feature_set_name
    predictions_df["feature_family"] = FEATURE_FAMILIES[feature_set_name]
    predictions_df["param_set"] = params["param_set"]
    predictions_df["score"] = scores
    predictions_df["prediction"] = preds
    predictions_df["decision_threshold"] = float(decision_threshold)
    return row, predictions_df


def select_params_on_validation(
    *,
    feature_set_name: str,
    train_input_df: pd.DataFrame,
    validation_input_df: pd.DataFrame,
    scope: str,
    ticker: str | None = None,
) -> tuple[dict, pd.DataFrame]:
    validation_rows = []
    for params in LOGREG_PARAM_GRID:
        validation_rows.append(
            evaluate_logreg_params(
                feature_set_name=feature_set_name,
                params=params,
                train_input_df=train_input_df,
                eval_input_df=validation_input_df,
                split_name="validation",
                scope=scope,
                ticker=ticker,
                tune_threshold=True,
            )
        )
    validation_results_df = pd.DataFrame(validation_rows)
    best_row = (
        validation_results_df.sort_values(
            ["balanced_accuracy", "auc", "accuracy", "param_set"],
            ascending=[False, False, False, True],
        )
        .head(1)
        .iloc[0]
        .to_dict()
    )
    return best_row, validation_results_df


def metrics_by_ticker_from_predictions(predictions_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for ticker, ticker_predictions_df in predictions_df.groupby("ticker"):
        metric_result = metrics_from_scores(
            ticker_predictions_df["target"],
            ticker_predictions_df["score"].to_numpy(),
            float(ticker_predictions_df["decision_threshold"].iloc[0]),
        )
        metric_result.pop("preds")
        rows.append(
            {
                "ticker": ticker,
                "feature_set": ticker_predictions_df["feature_set"].iloc[0],
                "feature_family": ticker_predictions_df["feature_family"].iloc[0],
                "is_attention_set": ticker_predictions_df["feature_set"].iloc[0] in ATTENTION_FEATURE_SETS,
                "param_set": ticker_predictions_df["param_set"].iloc[0],
                "test_rows": len(ticker_predictions_df),
                **metric_result,
            }
        )
    return pd.DataFrame(rows)


In [8]:
from __future__ import annotations

pooled_validation_rows = []
pooled_test_rows = []
pooled_prediction_frames = []
pooled_per_ticker_rows = []

for feature_set_name in FEATURE_SETS_TO_TEST:
    best_validation_row, validation_results_df = select_params_on_validation(
        feature_set_name=feature_set_name,
        train_input_df=train_df,
        validation_input_df=validation_df,
        scope="pooled",
    )
    pooled_validation_rows.extend(validation_results_df.to_dict(orient="records"))
    selected_params = next(
        params for params in LOGREG_PARAM_GRID if params["param_set"] == best_validation_row["param_set"]
    )
    test_row, predictions_df = evaluate_logreg_params(
        feature_set_name=feature_set_name,
        params=selected_params,
        train_input_df=train_df,
        eval_input_df=test_df,
        split_name="test",
        scope="pooled",
        decision_threshold=best_validation_row["decision_threshold"],
        return_predictions=True,
    )
    test_row["validation_balanced_accuracy"] = best_validation_row["balanced_accuracy"]
    test_row["validation_auc"] = best_validation_row["auc"]
    pooled_test_rows.append(test_row)
    pooled_prediction_frames.append(predictions_df)
    pooled_per_ticker_rows.append(metrics_by_ticker_from_predictions(predictions_df))

pooled_validation_results_df = pd.DataFrame(pooled_validation_rows)
pooled_feature_set_test_results_df = pd.DataFrame(pooled_test_rows).sort_values(
    ["balanced_accuracy", "auc", "accuracy", "feature_set"],
    ascending=[False, False, False, True],
).reset_index(drop=True)
pooled_test_predictions_df = pd.concat(pooled_prediction_frames, ignore_index=True)
pooled_per_ticker_test_results_df = pd.concat(pooled_per_ticker_rows, ignore_index=True)

pooled_baseline_by_ticker_df = pooled_per_ticker_test_results_df[
    pooled_per_ticker_test_results_df["feature_set"].eq(BASELINE_FEATURE_SET)
][["ticker", "accuracy", "balanced_accuracy", "auc"]].rename(
    columns={
        "accuracy": "baseline_accuracy",
        "balanced_accuracy": "baseline_balanced_accuracy",
        "auc": "baseline_auc",
    }
)

pooled_per_ticker_lift_vs_price_volume_df = pooled_per_ticker_test_results_df.merge(
    pooled_baseline_by_ticker_df,
    on="ticker",
    how="left",
)
pooled_per_ticker_lift_vs_price_volume_df["accuracy_lift_vs_price_volume"] = (
    pooled_per_ticker_lift_vs_price_volume_df["accuracy"]
    - pooled_per_ticker_lift_vs_price_volume_df["baseline_accuracy"]
)
pooled_per_ticker_lift_vs_price_volume_df["balanced_accuracy_lift_vs_price_volume"] = (
    pooled_per_ticker_lift_vs_price_volume_df["balanced_accuracy"]
    - pooled_per_ticker_lift_vs_price_volume_df["baseline_balanced_accuracy"]
)
pooled_per_ticker_lift_vs_price_volume_df["auc_lift_vs_price_volume"] = (
    pooled_per_ticker_lift_vs_price_volume_df["auc"] - pooled_per_ticker_lift_vs_price_volume_df["baseline_auc"]
)

pooled_feature_set_test_results_df


C:\Users\user\anaconda3\envs\equity-price-direction-predictor\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\user\anaconda3\envs\equity-price-direction-predictor\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\user\anaconda3\envs\equity-price-direction-predictor\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarnin

,scope,ticker,split,feature_set,feature_family,is_attention_set,features,param_set,n_features,train_rows,eval_rows,eval_positive_rate,decision_threshold,threshold_source,threshold_selection_balanced_accuracy,n_iter,n_nonzero_coefficients,predicted_positive_rate,accuracy,balanced_accuracy,auc,penalty,C,solver,class_weight,max_iter,validation_balanced_accuracy,validation_auc
0,pooled,ALL,test,price + volume + Reddit full,price + volume + Reddit,False,"return_1d, return_5d, return_20d, rolling_vola...",logreg_l2_C0p1_balanced,7,4456,1886,0.526511,0.057232,fixed,NaN,6,7,0.409862,0.518558,0.523403,0.528964,l2,0.1,lbfgs,balanced,3000,0.515213,0.504931
1,pooled,ALL,test,price + volume,price + volume,False,"return_1d, return_5d, return_20d, rolling_vola...",logreg_l2_C0p1_unweighted,5,4456,1886,0.526511,0.010827,fixed,NaN,6,5,0.739130,0.527572,0.514934,0.529601,l2,0.1,lbfgs,NaN,3000,0.511350,0.498747
2,pooled,ALL,test,price + volume + Reddit attention,price + volume + Reddit attention,True,"return_1d, return_5d, return_20d, rolling_vola...",logreg_l2_C0p1_balanced,6,4456,1886,0.526511,-0.029837,fixed,NaN,6,6,0.724284,0.522269,0.510407,0.530784,l2,0.1,lbfgs,balanced,3000,0.512453,0.501571
3,pooled,ALL,test,price only,price only,False,"return_1d, return_5d, return_20d, rolling_vola...",logreg_l1_C0p1_balanced,4,4456,1886,0.526511,-0.035445,fixed,NaN,8,3,0.754507,0.523860,0.510395,0.530141,l1,0.1,liblinear,balanced,3000,0.510798,0.497975
4,pooled,ALL,test,price + volume + GDELT attention,price + volume + GDELT attention,True,"return_1d, return_5d, return_20d, rolling_vola...",logreg_l1_C0p1_balanced,6,4456,1886,0.526511,-0.023128,fixed,NaN,6,4,0.677094,0.517497,0.508130,0.516482,l1,0.1,liblinear,balanced,3000,0.529037,0.506334
5,pooled,ALL,test,price + volume + GDELT full,price + volume + GDELT,False,"return_1d, return_5d, return_20d, rolling_vola...",logreg_l2_C0p1_balanced,7,4456,1886,0.526511,-0.023019,fixed,NaN,6,7,0.668081,0.516967,0.508078,0.516799,l2,0.1,lbfgs,balanced,3000,0.529589,0.506676
6,pooled,ALL,test,price + volume + all attention,price + volume + all attention,True,"return_1d, return_5d, return_20d, rolling_vola...",logreg_l1_C0p1_balanced,8,4456,1886,0.526511,-0.027793,fixed,NaN,7,5,0.691410,0.516967,0.506837,0.515274,l1,0.1,liblinear,balanced,3000,0.525623,0.505663
7,pooled,ALL,test,price + volume + alt sentiment compact,price + volume + compact alternative sentiment,False,"return_1d, return_5d, return_20d, rolling_vola...",logreg_l2_C0p1_balanced,8,4456,1886,0.526511,0.013525,fixed,NaN,6,8,0.562036,0.507423,0.504145,0.515879,l2,0.1,lbfgs,balanced,3000,0.515100,0.504338
8,pooled,ALL,test,price + volume + Google attention,price + volume + Google attention,True,"return_1d, return_5d, return_20d, rolling_vola...",logreg_l2_C0p1_balanced,6,4456,1886,0.526511,-0.012113,fixed,NaN,5,6,0.652174,0.511665,0.503606,0.514036,l2,0.1,lbfgs,balanced,3000,0.511827,0.496021


In [9]:
from __future__ import annotations

ticker_specific_validation_rows = []
ticker_specific_test_rows = []
skipped_ticker_models = []

for ticker in sorted(modeled_df["ticker"].unique()):
    ticker_train_df = train_df[train_df["ticker"].eq(ticker)].copy()
    ticker_validation_df = validation_df[validation_df["ticker"].eq(ticker)].copy()
    ticker_test_df = test_df[test_df["ticker"].eq(ticker)].copy()

    if (
        len(ticker_train_df) < CONFIG["min_ticker_train_rows"]
        or len(ticker_validation_df) < CONFIG["min_ticker_validation_rows"]
        or len(ticker_test_df) < CONFIG["min_ticker_test_rows"]
    ):
        skipped_ticker_models.append(
            {
                "ticker": ticker,
                "train_rows": len(ticker_train_df),
                "validation_rows": len(ticker_validation_df),
                "test_rows": len(ticker_test_df),
            }
        )
        continue

    for feature_set_name in FEATURE_SETS_TO_TEST:
        best_validation_row, validation_results_df = select_params_on_validation(
            feature_set_name=feature_set_name,
            train_input_df=ticker_train_df,
            validation_input_df=ticker_validation_df,
            scope="ticker_specific",
            ticker=ticker,
        )
        ticker_specific_validation_rows.extend(validation_results_df.to_dict(orient="records"))
        selected_params = next(
            params for params in LOGREG_PARAM_GRID if params["param_set"] == best_validation_row["param_set"]
        )
        test_row = evaluate_logreg_params(
            feature_set_name=feature_set_name,
            params=selected_params,
            train_input_df=ticker_train_df,
            eval_input_df=ticker_test_df,
            split_name="test",
            scope="ticker_specific",
            ticker=ticker,
            decision_threshold=best_validation_row["decision_threshold"],
        )
        test_row["validation_balanced_accuracy"] = best_validation_row["balanced_accuracy"]
        test_row["validation_auc"] = best_validation_row["auc"]
        ticker_specific_test_rows.append(test_row)

ticker_specific_validation_results_df = pd.DataFrame(ticker_specific_validation_rows)
ticker_specific_best_models_df = pd.DataFrame(ticker_specific_test_rows).sort_values(
    ["ticker", "balanced_accuracy", "auc", "accuracy", "feature_set"],
    ascending=[True, False, False, False, True],
).reset_index(drop=True)
skipped_ticker_models_df = pd.DataFrame(skipped_ticker_models)

ticker_baseline_df = ticker_specific_best_models_df[
    ticker_specific_best_models_df["feature_set"].eq(BASELINE_FEATURE_SET)
][["ticker", "accuracy", "balanced_accuracy", "auc"]].rename(
    columns={
        "accuracy": "baseline_accuracy",
        "balanced_accuracy": "baseline_balanced_accuracy",
        "auc": "baseline_auc",
    }
)

ticker_specific_lift_vs_price_volume_df = ticker_specific_best_models_df.merge(
    ticker_baseline_df,
    on="ticker",
    how="left",
)
ticker_specific_lift_vs_price_volume_df["accuracy_lift_vs_price_volume"] = (
    ticker_specific_lift_vs_price_volume_df["accuracy"]
    - ticker_specific_lift_vs_price_volume_df["baseline_accuracy"]
)
ticker_specific_lift_vs_price_volume_df["balanced_accuracy_lift_vs_price_volume"] = (
    ticker_specific_lift_vs_price_volume_df["balanced_accuracy"]
    - ticker_specific_lift_vs_price_volume_df["baseline_balanced_accuracy"]
)
ticker_specific_lift_vs_price_volume_df["auc_lift_vs_price_volume"] = (
    ticker_specific_lift_vs_price_volume_df["auc"] - ticker_specific_lift_vs_price_volume_df["baseline_auc"]
)

ticker_specific_best_models_df.head(20)


C:\Users\user\anaconda3\envs\equity-price-direction-predictor\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\user\anaconda3\envs\equity-price-direction-predictor\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\user\anaconda3\envs\equity-price-direction-predictor\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarnin

,scope,ticker,split,feature_set,feature_family,is_attention_set,features,param_set,n_features,train_rows,eval_rows,eval_positive_rate,decision_threshold,threshold_source,threshold_selection_balanced_accuracy,n_iter,n_nonzero_coefficients,predicted_positive_rate,accuracy,balanced_accuracy,auc,penalty,C,solver,class_weight,max_iter,validation_balanced_accuracy,validation_auc
0,ticker_specific,AAPL,test,price + volume + Reddit full,price + volume + Reddit,False,"return_1d, return_5d, return_20d, rolling_vola...",logreg_l1_C0p1_balanced,7,513,199,0.562814,0.016608,fixed,NaN,3,3,0.402010,0.537688,0.550800,0.539717,l1,0.1,liblinear,balanced,3000,0.555876,0.507896
1,ticker_specific,AAPL,test,price + volume + alt sentiment compact,price + volume + compact alternative sentiment,False,"return_1d, return_5d, return_20d, rolling_vola...",logreg_l1_C0p1_balanced,8,513,199,0.562814,0.002773,fixed,NaN,4,4,0.462312,0.537688,0.543103,0.546080,l1,0.1,liblinear,balanced,3000,0.564759,0.507440
2,ticker_specific,AAPL,test,price + volume + GDELT attention,price + volume + GDELT attention,True,"return_1d, return_5d, return_20d, rolling_vola...",logreg_l2_C0p1_balanced,6,513,199,0.562814,-0.260920,fixed,NaN,6,6,0.834171,0.567839,0.526273,0.495382,l2,0.1,lbfgs,balanced,3000,0.554130,0.495749
3,ticker_specific,AAPL,test,price only,price only,False,"return_1d, return_5d, return_20d, rolling_vola...",logreg_l2_C1_balanced,4,513,199,0.562814,0.162269,fixed,NaN,4,4,0.135678,0.472362,0.518422,0.532943,l2,1.0,lbfgs,balanced,3000,0.553750,0.485879
4,ticker_specific,AAPL,test,price + volume + Google attention,price + volume + Google attention,True,"return_1d, return_5d, return_20d, rolling_vola...",logreg_l1_C0p1_balanced,6,513,199,0.562814,-0.053772,fixed,NaN,3,2,0.919598,0.552764,0.500051,0.545567,l1,0.1,liblinear,balanced,3000,0.546234,0.495597
5,ticker_specific,AAPL,test,price + volume + Reddit attention,price + volume + Reddit attention,True,"return_1d, return_5d, return_20d, rolling_vola...",logreg_l1_C0p1_balanced,6,513,199,0.562814,-0.053772,fixed,NaN,3,2,0.919598,0.552764,0.500051,0.545567,l1,0.1,liblinear,balanced,3000,0.546234,0.495597
6,ticker_specific,AAPL,test,price + volume,price + volume,False,"return_1d, return_5d, return_20d, rolling_vola...",logreg_l1_C0p1_balanced,5,513,199,0.562814,-0.053768,fixed,NaN,2,2,0.919598,0.552764,0.500051,0.545464,l1,0.1,liblinear,balanced,3000,0.546234,0.495597
7,ticker_specific,AAPL,test,price + volume + all attention,price + volume + all attention,True,"return_1d, return_5d, return_20d, rolling_vola...",logreg_l2_C0p1_unweighted,8,513,199,0.562814,0.383024,fixed,NaN,6,8,0.070352,0.437186,0.491020,0.528633,l2,0.1,lbfgs,NaN,3000,0.546007,0.502885
8,ticker_specific,AAPL,test,price + volume + GDELT full,price + volume + GDELT,False,"return_1d, return_5d, return_20d, rolling_vola...",logreg_l1_C0p1_balanced,7,513,199,0.562814,-0.080755,fixed,NaN,4,4,0.939698,0.542714,0.487274,0.515702,l1,0.1,liblinear,balanced,3000,0.538339,0.504403
9,ticker_specific,AMD,test,price + volume + all attention,price + volume + all attention,True,"return_1d, return_5d, return_20d, rolling_vola...",logreg_l2_C1_balanced,8,585,254,0.515748,0.219753,fixed,NaN,7,8,0.173228,0.507874,0.518184,0.490598,l2,1.0,lbfgs,balanced,3000,0.549228,0.530972


In [10]:
from __future__ import annotations

def best_lift_rows(
    frame: pd.DataFrame,
    *,
    feature_filter: pd.Series,
    prefix: str,
) -> pd.DataFrame:
    best = (
        frame[feature_filter]
        .sort_values(
            ["ticker", "balanced_accuracy_lift_vs_price_volume", "auc_lift_vs_price_volume", "accuracy_lift_vs_price_volume"],
            ascending=[True, False, False, False],
        )
        .groupby("ticker", as_index=False)
        .head(1)
        .reset_index(drop=True)
    )
    return best[
        [
            "ticker",
            "feature_set",
            "feature_family",
            "balanced_accuracy_lift_vs_price_volume",
            "accuracy_lift_vs_price_volume",
            "auc_lift_vs_price_volume",
            "balanced_accuracy",
            "accuracy",
            "auc",
        ]
    ].rename(
        columns={
            "feature_set": f"{prefix}_best_feature_set",
            "feature_family": f"{prefix}_best_feature_family",
            "balanced_accuracy_lift_vs_price_volume": f"{prefix}_balanced_accuracy_lift",
            "accuracy_lift_vs_price_volume": f"{prefix}_accuracy_lift",
            "auc_lift_vs_price_volume": f"{prefix}_auc_lift",
            "balanced_accuracy": f"{prefix}_balanced_accuracy",
            "accuracy": f"{prefix}_accuracy",
            "auc": f"{prefix}_auc",
        }
    )


pooled_best_attention_lift_df = best_lift_rows(
    pooled_per_ticker_lift_vs_price_volume_df,
    feature_filter=pooled_per_ticker_lift_vs_price_volume_df["is_attention_set"],
    prefix="pooled_attention",
)
pooled_best_any_alternative_lift_df = best_lift_rows(
    pooled_per_ticker_lift_vs_price_volume_df,
    feature_filter=~pooled_per_ticker_lift_vs_price_volume_df["feature_family"].isin(["price only", "price + volume"]),
    prefix="pooled_any_alternative",
)
ticker_best_attention_lift_df = best_lift_rows(
    ticker_specific_lift_vs_price_volume_df,
    feature_filter=ticker_specific_lift_vs_price_volume_df["is_attention_set"],
    prefix="ticker_attention",
)
ticker_best_any_alternative_lift_df = best_lift_rows(
    ticker_specific_lift_vs_price_volume_df,
    feature_filter=~ticker_specific_lift_vs_price_volume_df["feature_family"].isin(["price only", "price + volume"]),
    prefix="ticker_any_alternative",
)

attention_coverage_model_lift_df = (
    attention_coverage_by_ticker_df[attention_coverage_display_columns]
    .merge(pooled_best_attention_lift_df, on="ticker", how="left")
    .merge(pooled_best_any_alternative_lift_df, on="ticker", how="left")
    .merge(ticker_best_attention_lift_df, on="ticker", how="left")
    .merge(ticker_best_any_alternative_lift_df, on="ticker", how="left")
    .sort_values(["low_overall_attention_flag", "avg_attention_rank"], ascending=[False, False])
    .reset_index(drop=True)
)

attention_coverage_model_lift_df


,ticker,rows,google_mean,reddit_mean,gdelt_mean,google_rank,reddit_rank,gdelt_rank,avg_attention_rank,weakest_attention_source,low_overall_attention_flag,pooled_attention_best_feature_set,pooled_attention_best_feature_family,pooled_attention_balanced_accuracy_lift,pooled_attention_accuracy_lift,pooled_attention_auc_lift,pooled_attention_balanced_accuracy,pooled_attention_accuracy,pooled_attention_auc,pooled_any_alternative_best_feature_set,pooled_any_alternative_best_feature_family,pooled_any_alternative_balanced_accuracy_lift,pooled_any_alternative_accuracy_lift,pooled_any_alternative_auc_lift,pooled_any_alternative_balanced_accuracy,pooled_any_alternative_accuracy,pooled_any_alternative_auc,ticker_attention_best_feature_set,ticker_attention_best_feature_family,ticker_attention_balanced_accuracy_lift,ticker_attention_accuracy_lift,ticker_attention_auc_lift,ticker_attention_balanced_accuracy,ticker_attention_accuracy,ticker_attention_auc,ticker_any_alternative_best_feature_set,ticker_any_alternative_best_feature_family,ticker_any_alternative_balanced_accuracy_lift,ticker_any_alternative_accuracy_lift,ticker_any_alternative_auc_lift,ticker_any_alternative_balanced_accuracy,ticker_any_alternative_accuracy,ticker_any_alternative_auc
0,AMD,1255,21.512428,30.303586,0.020628,7.0,7.0,8.0,7.333333,gdelt,True,price + volume + GDELT attention,price + volume + GDELT attention,0.009123,0.007874,0.006641,0.502948,0.503937,0.509402,price + volume + Reddit full,price + volume + Reddit,0.037299,0.023622,-0.015391,0.531124,0.519685,0.487370,price + volume + all attention,price + volume + all attention,0.043009,0.023622,0.007509,0.518184,0.507874,0.490598,price + volume + all attention,price + volume + all attention,0.043009,0.023622,0.007509,0.518184,0.507874,0.490598
1,NVDA,1255,17.126982,58.322709,0.158757,8.0,4.0,5.0,5.666667,google,True,price + volume + GDELT attention,price + volume + GDELT attention,0.008053,-0.003891,0.003491,0.546509,0.556420,0.558849,price + volume + GDELT attention,price + volume + GDELT attention,0.008053,-0.003891,0.003491,0.546509,0.556420,0.558849,price + volume + GDELT attention,price + volume + GDELT attention,0.018187,0.000000,0.007471,0.526822,0.529183,0.535150,price + volume + Reddit full,price + volume + Reddit,0.019167,-0.027237,0.003735,0.527802,0.501946,0.531415
2,AMZN,1255,31.598097,54.796813,0.085730,4.0,5.0,6.0,5.000000,gdelt,False,price + volume + all attention,price + volume + all attention,0.009964,0.008511,-0.023188,0.516667,0.523404,0.509275,price + volume + Reddit full,price + volume + Reddit,0.020290,0.012766,0.006667,0.526993,0.527660,0.539130,price + volume + Google attention,price + volume + Google attention,-0.001993,-0.004255,-0.000072,0.489855,0.480851,0.513116,price + volume + alt sentiment compact,price + volume + compact alternative sentiment,0.027174,0.029787,-0.020435,0.519022,0.514894,0.492754
3,META,1255,22.519854,24.498805,3.054745,6.0,8.0,1.0,5.000000,reddit,False,price + volume + Reddit attention,price + volume + Reddit attention,0.004383,0.004274,0.003872,0.539743,0.542735,0.528419,price + volume + Reddit attention,price + volume + Reddit attention,0.004383,0.004274,0.003872,0.539743,0.542735,0.528419,price + volume + GDELT attention,price + volume + GDELT attention,0.025497,0.025641,0.017168,0.510082,0.508547,0.528419,price + volume + alt sentiment compact,price + volume + compact alternative sentiment,0.070061,0.072650,0.039524,0.554646,0.555556,0.550774
4,AAPL,1255,35.245331,81.182470,0.045539,2.0,2.0,7.0,3.666667,gdelt,False,price + volume + Reddit attention,price + volume + Reddit attention,0.005747,0.005025,0.000308,0.480911,0.532663,0.516010,price + volume + Reddit full,price + volume + Reddit,0.012264,-0.015075,0.013239,0.487428,0.512563,0.528941,price + volume + GDELT attention,price + volume + GDELT attention,0.026221,0.015075,-0.050082,0.526273,0.567839,0.495382,price + volume + Reddit full,price + volume + Reddit,0.050749,-0.015075,-0.005747,0.550800,0.537688,0.5397

In [11]:
from __future__ import annotations

correlation_metric_pairs = [
    ("google_mean", "Google mean attention"),
    ("reddit_mean", "Reddit mean attention"),
    ("gdelt_mean", "GDELT mean attention"),
    ("avg_attention_rank", "Average attention rank; higher means weaker attention"),
]
lift_metric_pairs = [
    ("pooled_attention_balanced_accuracy_lift", "Pooled model attention lift"),
    ("pooled_any_alternative_balanced_accuracy_lift", "Pooled model any-alt lift"),
    ("ticker_attention_balanced_accuracy_lift", "Ticker-specific attention lift"),
    ("ticker_any_alternative_balanced_accuracy_lift", "Ticker-specific any-alt lift"),
]

correlation_rows = []
for coverage_column, coverage_label in correlation_metric_pairs:
    for lift_column, lift_label in lift_metric_pairs:
        valid_df = attention_coverage_model_lift_df[[coverage_column, lift_column]].dropna()
        correlation_rows.append(
            {
                "coverage_metric": coverage_label,
                "lift_metric": lift_label,
                "n_tickers": len(valid_df),
                "pearson_correlation": valid_df[coverage_column].corr(valid_df[lift_column], method="pearson"),
                "spearman_correlation": valid_df[coverage_column].corr(valid_df[lift_column], method="spearman"),
            }
        )

coverage_lift_correlation_df = pd.DataFrame(correlation_rows)

coverage_lift_correlation_df


,coverage_metric,lift_metric,n_tickers,pearson_correlation,spearman_correlation
0,Google mean attention,Pooled model attention lift,8,0.295011,0.404762
1,Google mean attention,Pooled model any-alt lift,8,0.014869,0.142857
2,Google mean attention,Ticker-specific attention lift,8,-0.530591,-0.285714
3,Google mean attention,Ticker-specific any-alt lift,8,-0.115980,0.166667
4,Reddit mean attention,Pooled model attention lift,8,-0.600156,-0.190476
5,Reddit mean attention,Pooled model any-alt lift,8,-0.433691,-0.190476
6,Reddit mean attention,Ticker-specific attention lift,8,-0.154790,-0.166667
7,Reddit mean attention,Ticker-specific any-alt lift,8,-0.557525,-0.619048
8,GDELT mean attention,Pooled model attention lift,8,-0.138492,0.047619
9,GDELT mean attention,Pooled model any-alt lift,8,-0.302806,-0.476190


In [12]:
from __future__ import annotations

def concern_label(row: pd.Series) -> str:
    lift = row["ticker_attention_balanced_accuracy_lift"]
    if pd.isna(lift):
        return "not enough data for ticker-specific conclusion"
    if row["low_overall_attention_flag"] and lift <= 0:
        return "supports concern: low attention and no attention lift"
    if row["low_overall_attention_flag"] and lift > 0:
        return "low attention but attention still helped"
    if not row["low_overall_attention_flag"] and lift <= 0:
        return "attention available but no attention lift"
    return "attention available and attention helped"


concern_summary_columns = [
    "ticker",
    "low_overall_attention_flag",
    "weakest_attention_source",
    "google_mean",
    "reddit_mean",
    "gdelt_mean",
    "avg_attention_rank",
    "ticker_attention_best_feature_set",
    "ticker_attention_balanced_accuracy_lift",
    "ticker_any_alternative_best_feature_set",
    "ticker_any_alternative_balanced_accuracy_lift",
    "pooled_attention_best_feature_set",
    "pooled_attention_balanced_accuracy_lift",
]

concern_summary_df = attention_coverage_model_lift_df[concern_summary_columns].copy()
concern_summary_df["diagnostic_label"] = concern_summary_df.apply(concern_label, axis=1)
concern_summary_df = concern_summary_df.sort_values(
    ["low_overall_attention_flag", "ticker_attention_balanced_accuracy_lift"],
    ascending=[False, True],
).reset_index(drop=True)

concern_summary_df


,ticker,low_overall_attention_flag,weakest_attention_source,google_mean,reddit_mean,gdelt_mean,avg_attention_rank,ticker_attention_best_feature_set,ticker_attention_balanced_accuracy_lift,ticker_any_alternative_best_feature_set,ticker_any_alternative_balanced_accuracy_lift,pooled_attention_best_feature_set,pooled_attention_balanced_accuracy_lift,diagnostic_label
0,NVDA,True,google,17.126982,58.322709,0.158757,5.666667,price + volume + GDELT attention,0.018187,price + volume + Reddit full,0.019167,price + volume + GDELT attention,0.008053,low attention but attention still helped
1,AMD,True,gdelt,21.512428,30.303586,0.020628,7.333333,price + volume + all attention,0.043009,price + volume + all attention,0.043009,price + volume + GDELT attention,0.009123,low attention but attention still helped
2,AMZN,False,gdelt,31.598097,54.796813,0.085730,5.000000,price + volume + Google attention,-0.001993,price + volume + alt sentiment compact,0.027174,price + volume + all attention,0.009964,attention available but no attention lift
3,MSFT,False,reddit,43.078871,48.926693,0.567273,3.333333,price + volume + GDELT attention,0.000000,price + volume + GDELT full,0.028986,price + volume + Google attention,0.010337,attention available but no attention lift
4,TSLA,False,google,26.664668,122.027888,0.554015,3.333333,price + volume + Google attention,0.017978,price + volume + Google attention,0.017978,price + volume + all attention,-0.003163,attention available and attention helped
5,GOOGL,False,google,34.146333,70.502789,1.337674,2.666667,price + volume + Reddit attention,0.023218,price + volume + Reddit attention,0.023218,price + volume + Reddit attention,0.012148,attention available and attention helped
6,META,False,reddit,22.519854,24.498805,3.054745,5.000000,price + volume + GDELT attention,0.025497,price + volume + alt sentiment compact,0.070061,price + volume + Reddit attention,0.004383,attention available and attention helped
7,AAPL,False,gdelt,35.245331,81.182470,0.045539,3.666667,price + volume + GDELT attention,0.026221,price + volume + Reddit full,0.050749,price + volume + Reddit attention,0.005747,attention available and attention helped


## Basic Model Family Check

This section uses a deliberately small set of basic models to verify whether alternative-data feature sets improve test balanced accuracy versus `price + volume`. These are diagnostics, not a broad tuning grid.


In [13]:
from __future__ import annotations

BASIC_MODEL_CONFIGS = [
    {
        "model_name": "dummy_most_frequent",
        "model_family": "dummy",
        "default_threshold": 0.5,
        "tune_threshold": False,
    },
    {
        "model_name": "logreg_l2_C0p1_balanced",
        "model_family": "logistic_regression",
        "default_threshold": 0.0,
        "tune_threshold": True,
        "model_params": {
            "penalty": "l2",
            "C": 0.1,
            "solver": "lbfgs",
            "class_weight": "balanced",
            "max_iter": 3000,
            "random_state": CONFIG["random_state"],
        },
    },
    {
        "model_name": "linear_svm_C0p1_balanced",
        "model_family": "linear_svm",
        "default_threshold": 0.0,
        "tune_threshold": True,
        "model_params": {
            "C": 0.1,
            "class_weight": "balanced",
            "dual": False,
            "max_iter": 5000,
            "random_state": CONFIG["random_state"],
        },
    },
    {
        "model_name": "rf_shallow_balanced",
        "model_family": "random_forest",
        "default_threshold": 0.5,
        "tune_threshold": True,
        "model_params": {
            "n_estimators": 300,
            "max_depth": 4,
            "min_samples_leaf": 25,
            "min_samples_split": 50,
            "max_features": "sqrt",
            "class_weight": "balanced_subsample",
            "bootstrap": True,
            "n_jobs": -1,
            "random_state": CONFIG["random_state"],
        },
    },
]


def build_basic_model_pipeline(model_config: dict) -> Pipeline:
    model_family = model_config["model_family"]
    if model_family == "dummy":
        return Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("model", DummyClassifier(strategy="most_frequent")),
            ]
        )
    if model_family == "logistic_regression":
        return Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(**model_config["model_params"])),
            ]
        )
    if model_family == "linear_svm":
        return Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", LinearSVC(**model_config["model_params"])),
            ]
        )
    if model_family == "random_forest":
        return Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("model", RandomForestClassifier(**model_config["model_params"])),
            ]
        )
    raise ValueError(f"Unsupported model family: {model_family}")


def score_basic_model_pipeline(pipeline: Pipeline, features: list[str], frame: pd.DataFrame) -> np.ndarray:
    x_values = frame[features]
    if hasattr(pipeline, "decision_function"):
        return np.asarray(pipeline.decision_function(x_values), dtype=float)
    if hasattr(pipeline, "predict_proba"):
        probabilities = pipeline.predict_proba(x_values)
        if probabilities.shape[1] == 1:
            predicted_class = pipeline.classes_[0]
            return np.ones(len(frame), dtype=float) if predicted_class == 1 else np.zeros(len(frame), dtype=float)
        positive_class_index = list(pipeline.classes_).index(1)
        return np.asarray(probabilities[:, positive_class_index], dtype=float)
    return np.asarray(pipeline.predict(x_values), dtype=float)


def evaluate_basic_model_feature_set(
    *,
    model_config: dict,
    feature_set_name: str,
    train_input_df: pd.DataFrame,
    validation_input_df: pd.DataFrame,
    test_input_df: pd.DataFrame,
) -> tuple[dict, pd.DataFrame]:
    features = FEATURE_SETS[feature_set_name]
    pipeline = build_basic_model_pipeline(model_config)
    pipeline.fit(train_input_df[features], train_input_df["target"])

    validation_scores = score_basic_model_pipeline(pipeline, features, validation_input_df)
    if model_config["tune_threshold"]:
        decision_threshold, threshold_selection_balanced_accuracy = best_threshold_for_balanced_accuracy(
            validation_input_df["target"],
            validation_scores,
        )
        threshold_source = "validation_tuned"
    else:
        decision_threshold = float(model_config["default_threshold"])
        threshold_selection_balanced_accuracy = np.nan
        threshold_source = "default"

    validation_metrics = metrics_from_scores(validation_input_df["target"], validation_scores, float(decision_threshold))
    validation_metrics.pop("preds")

    test_scores = score_basic_model_pipeline(pipeline, features, test_input_df)
    test_metrics = metrics_from_scores(test_input_df["target"], test_scores, float(decision_threshold))
    test_predictions = test_metrics.pop("preds")

    row = {
        "model_name": model_config["model_name"],
        "model_family": model_config["model_family"],
        "feature_set": feature_set_name,
        "feature_family": FEATURE_FAMILIES[feature_set_name],
        "is_attention_set": feature_set_name in ATTENTION_FEATURE_SETS,
        "n_features": len(features),
        "features": ", ".join(features),
        "train_rows": len(train_input_df),
        "validation_rows": len(validation_input_df),
        "test_rows": len(test_input_df),
        "decision_threshold": float(decision_threshold),
        "threshold_source": threshold_source,
        "threshold_selection_balanced_accuracy": threshold_selection_balanced_accuracy,
        "validation_accuracy": validation_metrics["accuracy"],
        "validation_balanced_accuracy": validation_metrics["balanced_accuracy"],
        "validation_auc": validation_metrics["auc"],
        "validation_predicted_positive_rate": validation_metrics["predicted_positive_rate"],
        "test_accuracy": test_metrics["accuracy"],
        "test_balanced_accuracy": test_metrics["balanced_accuracy"],
        "test_auc": test_metrics["auc"],
        "test_predicted_positive_rate": test_metrics["predicted_positive_rate"],
    }

    predictions_df = test_input_df[["date", "ticker", "target"]].copy()
    predictions_df["model_name"] = model_config["model_name"]
    predictions_df["model_family"] = model_config["model_family"]
    predictions_df["feature_set"] = feature_set_name
    predictions_df["feature_family"] = FEATURE_FAMILIES[feature_set_name]
    predictions_df["is_attention_set"] = feature_set_name in ATTENTION_FEATURE_SETS
    predictions_df["score"] = test_scores
    predictions_df["prediction"] = test_predictions
    predictions_df["decision_threshold"] = float(decision_threshold)
    return row, predictions_df


basic_model_rows = []
basic_model_prediction_frames = []

for model_config in BASIC_MODEL_CONFIGS:
    for feature_set_name in FEATURE_SETS_TO_TEST:
        result_row, predictions_df = evaluate_basic_model_feature_set(
            model_config=model_config,
            feature_set_name=feature_set_name,
            train_input_df=train_df,
            validation_input_df=validation_df,
            test_input_df=test_df,
        )
        basic_model_rows.append(result_row)
        basic_model_prediction_frames.append(predictions_df)

basic_model_feature_set_test_results_df = pd.DataFrame(basic_model_rows).sort_values(
    ["model_name", "test_balanced_accuracy", "test_auc", "test_accuracy", "feature_set"],
    ascending=[True, False, False, False, True],
).reset_index(drop=True)
basic_model_test_predictions_df = pd.concat(basic_model_prediction_frames, ignore_index=True)

basic_model_feature_set_test_results_df


C:\Users\user\anaconda3\envs\equity-price-direction-predictor\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\user\anaconda3\envs\equity-price-direction-predictor\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\user\anaconda3\envs\equity-price-direction-predictor\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarnin

,model_name,model_family,feature_set,feature_family,is_attention_set,n_features,features,train_rows,validation_rows,test_rows,decision_threshold,threshold_source,threshold_selection_balanced_accuracy,validation_accuracy,validation_balanced_accuracy,validation_auc,validation_predicted_positive_rate,test_accuracy,test_balanced_accuracy,test_auc,test_predicted_positive_rate
0,dummy_most_frequent,dummy,price + volume,price + volume,False,5,"return_1d, return_5d, return_20d, rolling_vola...",4456,1424,1886,0.500000,default,NaN,0.568118,0.500000,0.500000,1.000000,0.526511,0.500000,0.500000,1.000000
1,dummy_most_frequent,dummy,price + volume + GDELT attention,price + volume + GDELT attention,True,6,"return_1d, return_5d, return_20d, rolling_vola...",4456,1424,1886,0.500000,default,NaN,0.568118,0.500000,0.500000,1.000000,0.526511,0.500000,0.500000,1.000000
2,dummy_most_frequent,dummy,price + volume + GDELT full,price + volume + GDELT,False,7,"return_1d, return_5d, return_20d, rolling_vola...",4456,1424,1886,0.500000,default,NaN,0.568118,0.500000,0.500000,1.000000,0.526511,0.500000,0.500000,1.000000
3,dummy_most_frequent,dummy,price + volume + Google attention,price + volume + Google attention,True,6,"return_1d, return_5d, return_20d, rolling_vola...",4456,1424,1886,0.500000,default,NaN,0.568118,0.500000,0.500000,1.000000,0.526511,0.500000,0.500000,1.000000
4,dummy_most_frequent,dummy,price + volume + Reddit attention,price + volume + Reddit attention,True,6,"return_1d, return_5d, return_20d, rolling_vola...",4456,1424,1886,0.500000,default,NaN,0.568118,0.500000,0.500000,1.000000,0.526511,0.500000,0.500000,1.000000
5,dummy_most_frequent,dummy,price + volume + Reddit full,price + volume + Reddit,False,7,"return_1d, return_5d, return_20d, rolling_vola...",4456,1424,1886,0.500000,default,NaN,0.568118,0.500000,0.500000,1.000000,0.526511,0.500000,0.500000,1.000000
6,dummy_most_frequent,dummy,price + volume + all attention,price + volume + all attention,True,8,"return_1d, return_5d, return_20d, rolling_vola...",4456,1424,1886,0.500000,default,NaN,0.568118,0.500000,0.500000,1.000000,0.526511,0.500000,0.500000,1.000000
7,dummy_most_frequent,dummy,price + volume + alt sentiment compact,price + volume + compact alternative sentiment,False,8,"return_1d, return_5d, return_20d, rolling_vola...",4456,1424,1886,0.500000,default,NaN,0.568118,0.500000,0.500000,1.000000,0.526511,0.500000,0.500000,1.000000
8,dummy_most_frequent,dummy,price only,price only,False,4,"return_1d, return_5d, return_20d, rolling_vola...",4456,1424,1886,0.500000,default,NaN,0.568118,0.500000,0.500000,1.000000,0.526511,0.500000,0.500000,1.000000
9,linear_svm_C0p1_balanced,linear_svm,price + volume + Reddit full,price + volume + Reddit,False,7,"return_1d, return_5d, return_20d, rolling_vola...",4456,1424,1886,0.028983,validation_tuned,0.513782,0.504916,0.513782,0.504724,0.436798,0.519618,0.524523,0.528934,0.408802


In [14]:
from __future__ import annotations

basic_model_baseline_df = basic_model_feature_set_test_results_df[
    basic_model_feature_set_test_results_df["feature_set"].eq(BASELINE_FEATURE_SET)
][
    [
        "model_name",
        "test_accuracy",
        "test_balanced_accuracy",
        "test_auc",
        "validation_balanced_accuracy",
        "validation_auc",
    ]
].rename(
    columns={
        "test_accuracy": "baseline_test_accuracy",
        "test_balanced_accuracy": "baseline_test_balanced_accuracy",
        "test_auc": "baseline_test_auc",
        "validation_balanced_accuracy": "baseline_validation_balanced_accuracy",
        "validation_auc": "baseline_validation_auc",
    }
)

basic_model_alternative_lift_df = basic_model_feature_set_test_results_df.merge(
    basic_model_baseline_df,
    on="model_name",
    how="left",
)
basic_model_alternative_lift_df["test_accuracy_lift_vs_price_volume"] = (
    basic_model_alternative_lift_df["test_accuracy"] - basic_model_alternative_lift_df["baseline_test_accuracy"]
)
basic_model_alternative_lift_df["test_balanced_accuracy_lift_vs_price_volume"] = (
    basic_model_alternative_lift_df["test_balanced_accuracy"]
    - basic_model_alternative_lift_df["baseline_test_balanced_accuracy"]
)
basic_model_alternative_lift_df["test_auc_lift_vs_price_volume"] = (
    basic_model_alternative_lift_df["test_auc"] - basic_model_alternative_lift_df["baseline_test_auc"]
)
basic_model_alternative_lift_df["validation_balanced_accuracy_lift_vs_price_volume"] = (
    basic_model_alternative_lift_df["validation_balanced_accuracy"]
    - basic_model_alternative_lift_df["baseline_validation_balanced_accuracy"]
)

basic_model_validation_selected_alternative_df = (
    basic_model_alternative_lift_df[
        ~basic_model_alternative_lift_df["feature_family"].isin(["price only", "price + volume"])
    ]
    .sort_values(
        ["model_name", "validation_balanced_accuracy", "validation_auc", "test_balanced_accuracy"],
        ascending=[True, False, False, False],
    )
    .groupby("model_name", as_index=False)
    .head(1)
    .sort_values("test_balanced_accuracy_lift_vs_price_volume", ascending=False)
    .reset_index(drop=True)
)

basic_model_best_test_alternative_df = (
    basic_model_alternative_lift_df[
        ~basic_model_alternative_lift_df["feature_family"].isin(["price only", "price + volume"])
    ]
    .sort_values(
        ["model_name", "test_balanced_accuracy_lift_vs_price_volume", "test_auc_lift_vs_price_volume"],
        ascending=[True, False, False],
    )
    .groupby("model_name", as_index=False)
    .head(1)
    .sort_values("test_balanced_accuracy_lift_vs_price_volume", ascending=False)
    .reset_index(drop=True)
)

basic_model_alternative_data_question_df = basic_model_validation_selected_alternative_df[
    [
        "model_name",
        "model_family",
        "feature_set",
        "feature_family",
        "is_attention_set",
        "validation_balanced_accuracy",
        "baseline_validation_balanced_accuracy",
        "validation_balanced_accuracy_lift_vs_price_volume",
        "test_balanced_accuracy",
        "baseline_test_balanced_accuracy",
        "test_balanced_accuracy_lift_vs_price_volume",
        "test_accuracy",
        "baseline_test_accuracy",
        "test_accuracy_lift_vs_price_volume",
        "test_auc",
        "baseline_test_auc",
        "test_auc_lift_vs_price_volume",
    ]
].copy()
basic_model_alternative_data_question_df["helps_test_balanced_accuracy"] = (
    basic_model_alternative_data_question_df["test_balanced_accuracy_lift_vs_price_volume"] > 0
)


def basic_metrics_by_ticker(predictions_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (model_name, feature_set, ticker), group in predictions_df.groupby(["model_name", "feature_set", "ticker"]):
        metric_result = metrics_from_scores(
            group["target"],
            group["score"].to_numpy(),
            float(group["decision_threshold"].iloc[0]),
        )
        metric_result.pop("preds")
        rows.append(
            {
                "model_name": model_name,
                "model_family": group["model_family"].iloc[0],
                "ticker": ticker,
                "feature_set": feature_set,
                "feature_family": group["feature_family"].iloc[0],
                "is_attention_set": bool(group["is_attention_set"].iloc[0]),
                "test_rows": len(group),
                "test_accuracy": metric_result["accuracy"],
                "test_balanced_accuracy": metric_result["balanced_accuracy"],
                "test_auc": metric_result["auc"],
                "test_predicted_positive_rate": metric_result["predicted_positive_rate"],
            }
        )
    return pd.DataFrame(rows)


basic_model_per_ticker_results_df = basic_metrics_by_ticker(basic_model_test_predictions_df)
basic_model_per_ticker_baseline_df = basic_model_per_ticker_results_df[
    basic_model_per_ticker_results_df["feature_set"].eq(BASELINE_FEATURE_SET)
][["model_name", "ticker", "test_accuracy", "test_balanced_accuracy", "test_auc"]].rename(
    columns={
        "test_accuracy": "baseline_test_accuracy",
        "test_balanced_accuracy": "baseline_test_balanced_accuracy",
        "test_auc": "baseline_test_auc",
    }
)
basic_model_per_ticker_lift_df = basic_model_per_ticker_results_df.merge(
    basic_model_per_ticker_baseline_df,
    on=["model_name", "ticker"],
    how="left",
)
basic_model_per_ticker_lift_df["test_balanced_accuracy_lift_vs_price_volume"] = (
    basic_model_per_ticker_lift_df["test_balanced_accuracy"]
    - basic_model_per_ticker_lift_df["baseline_test_balanced_accuracy"]
)
basic_model_per_ticker_lift_df["test_accuracy_lift_vs_price_volume"] = (
    basic_model_per_ticker_lift_df["test_accuracy"] - basic_model_per_ticker_lift_df["baseline_test_accuracy"]
)
basic_model_per_ticker_lift_df["test_auc_lift_vs_price_volume"] = (
    basic_model_per_ticker_lift_df["test_auc"] - basic_model_per_ticker_lift_df["baseline_test_auc"]
)

basic_model_ticker_best_alternative_df = (
    basic_model_per_ticker_lift_df[
        ~basic_model_per_ticker_lift_df["feature_family"].isin(["price only", "price + volume"])
    ]
    .sort_values(
        ["model_name", "ticker", "test_balanced_accuracy_lift_vs_price_volume", "test_auc_lift_vs_price_volume"],
        ascending=[True, True, False, False],
    )
    .groupby(["model_name", "ticker"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

basic_model_alternative_data_question_df


,model_name,model_family,feature_set,feature_family,is_attention_set,validation_balanced_accuracy,baseline_validation_balanced_accuracy,validation_balanced_accuracy_lift_vs_price_volume,test_balanced_accuracy,baseline_test_balanced_accuracy,test_balanced_accuracy_lift_vs_price_volume,test_accuracy,baseline_test_accuracy,test_accuracy_lift_vs_price_volume,test_auc,baseline_test_auc,test_auc_lift_vs_price_volume,helps_test_balanced_accuracy
0,rf_shallow_balanced,random_forest,price + volume + GDELT full,price + volume + GDELT,False,0.527643,0.511434,0.016209,0.523385,0.520333,0.003052,0.536055,0.514316,0.021739,0.532728,0.538300,-0.005572,True
1,dummy_most_frequent,dummy,price + volume + GDELT attention,price + volume + GDELT attention,True,0.500000,0.500000,0.000000,0.500000,0.500000,0.000000,0.526511,0.526511,0.000000,0.500000,0.500000,0.000000,False
2,linear_svm_C0p1_balanced,linear_svm,price + volume + GDELT full,price + volume + GDELT,False,0.528158,0.509919,0.018239,0.508078,0.514878,-0.006800,0.516967,0.527572,-0.010604,0.516838,0.529512,-0.012673,False
3,logreg_l2_C0p1_balanced,logistic_regression,price + volume + GDELT full,price + volume + GDELT,False,0.529589,0.509919,0.019670,0.508078,0.514934,-0.006857,0.516967,0.527572,-0.010604,0.516799,0.529600,-0.012801,False


In [15]:
from __future__ import annotations

basic_model_ticker_help_summary_df = (
    basic_model_ticker_best_alternative_df.assign(
        helps_test_balanced_accuracy=lambda frame: frame["test_balanced_accuracy_lift_vs_price_volume"] > 0
    )
    .groupby(["model_name", "model_family"])
    .agg(
        tickers_with_positive_alt_lift=("helps_test_balanced_accuracy", "sum"),
        tickers_evaluated=("ticker", "nunique"),
        mean_best_alt_balanced_accuracy_lift=("test_balanced_accuracy_lift_vs_price_volume", "mean"),
        median_best_alt_balanced_accuracy_lift=("test_balanced_accuracy_lift_vs_price_volume", "median"),
        best_single_ticker_lift=("test_balanced_accuracy_lift_vs_price_volume", "max"),
        worst_single_ticker_lift=("test_balanced_accuracy_lift_vs_price_volume", "min"),
    )
    .reset_index()
)
basic_model_ticker_help_summary_df["share_tickers_with_positive_alt_lift"] = (
    basic_model_ticker_help_summary_df["tickers_with_positive_alt_lift"]
    / basic_model_ticker_help_summary_df["tickers_evaluated"]
)

basic_model_ticker_attention_concern_df = (
    basic_model_ticker_best_alternative_df.merge(
        attention_coverage_by_ticker_df[
            ["ticker", "avg_attention_rank", "low_overall_attention_flag", "weakest_attention_source"]
        ],
        on="ticker",
        how="left",
    )
    .sort_values(["model_name", "low_overall_attention_flag", "test_balanced_accuracy_lift_vs_price_volume"], ascending=[True, False, True])
    .reset_index(drop=True)
)

basic_model_ticker_help_summary_df


,model_name,model_family,tickers_with_positive_alt_lift,tickers_evaluated,mean_best_alt_balanced_accuracy_lift,median_best_alt_balanced_accuracy_lift,best_single_ticker_lift,worst_single_ticker_lift,share_tickers_with_positive_alt_lift
0,dummy_most_frequent,dummy,0,8,0.000000,0.000000,0.000000,0.000000,0.000
1,linear_svm_C0p1_balanced,linear_svm,8,8,0.015859,0.011689,0.033482,0.004383,1.000
2,logreg_l2_C0p1_balanced,logistic_regression,7,8,0.014559,0.011865,0.037299,-0.000973,0.875
3,rf_shallow_balanced,random_forest,6,8,0.029280,0.024117,0.078942,-0.029515,0.750
